Huấn Luyện Gradient Boosted Trees Regressor Hệ thống Dự báo tắc nghẽn Giao thông. Gradient Boosted Trees Regressor (Trình hồi quy Cây tăng cường Gradient)

### Tải một số thư viện

In [ ]:
!pip install pyspark findspark folium pandas numpy matplotlib seaborn scipy
!pip install pyngrok
!pip install rainbow-tqdm


### Kết nối Google Drive
Cell dưới đây sẽ kết nối (Mount) Google Drive của bạn trực tiếp vào Colab và trỏ thư mục làm việc vào thư mục `Traffic_Project` trên `MyDrive`.
Khi đó, mọi kết quả huấn luyện (biểu đồ, model, CSV phân tích) sẽ được lưu thẳng về bộ nhớ Google Drive của bạn chứ không bị mất đi.

In [ ]:
from google.colab import drive
import os
# 1. Kết nối Google Drive
drive.mount('/content/drive')
# 2. Đường dẫn đến tệp train.csv đã có trên Drive
data_dir = '/content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/Data'
train_csv_path = os.path.join(data_dir, 'train.csv')
# 3. Kiểm tra sự tồn tại của tệp và thông báo
if os.path.exists(train_csv_path):
    print(f"[OK] Đã tìm thấy tệp dữ liệu tại: {train_csv_path}")
    # Kiểm tra kích thước tệp
    file_size = os.path.getsize(train_csv_path) / (1024 * 1024)
    print(f"[INFO] Kích thước tệp: {file_size:.2f} MB")
else:
    print(f"[LỖI] Không tìm thấy tệp train.csv tại: {train_csv_path}")
    print("[GỢI Ý] Vui lòng kiểm tra lại đường dẫn hoặc đảm bảo tệp đã được bỏ vào thư mục Data.")

Mounted at /content/drive
[OK] Đã tìm thấy tệp dữ liệu tại: /content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/Data/train.csv
[INFO] Kích thước tệp: 191.30 MB


### Hệ thống dùng để huấn luyện dự báo mô hình

In [ ]:
# 1. Thư viện hệ thống và tiêu chuẩn
import os
import sys
import json
import time
import math
import gc
import threading
# 2. Xử lý dữ liệu & tính toán khoa học
import pandas as pd
import numpy as np
from scipy.stats import norm
import scipy.stats as stats
import statsmodels.api as sm
# 3. Trực quan hóa dữ liệu & bản đồ
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap, HeatMapWithTime, DualMap
from IPython.display import display, HTML
# 4. Tiện ích mở rộng (ngrok, tqdm)
from pyngrok import ngrok
from rainbow_tqdm import tqdm
# 5. PySpark: Core & Cấu hình
from pyspark.sql import SparkSession
from pyspark import StorageLevel
# 6. PySpark: SQL Functions (Gom nhóm toàn bộ hàm)
from pyspark.sql.functions import (
    col, expr, exp, sin, cos, radians, asin, sqrt, desc,
    abs as spark_abs,
    mean as spark_mean,
    round as spark_round,
    stddev as spark_stddev,  # Tính độ lệch chuẩn cho biểu đồ EDA số 2
    to_timestamp,            # Ép kiểu dữ liệu thời gian lúc mới đọc CSV
    when, lit                # Cần thiết để xử lý logic điều kiện (VD: safe_pred_speed)
)
# 7. PySpark: Machine Learning (ML)
from pyspark.ml.feature import VectorAssembler, StringIndexer, VectorIndexer, OneHotEncoder
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.stat import Correlation
from pyspark.ml.stat import Correlation

# Khắc phục lỗi in tiếng Việt trên Windows/Colab
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
    sys.stderr.reconfigure(encoding='utf-8')

# ================================================================
# BANNER HỆ THỐNG
# ================================================================
print("")
print("================================================================")
print("  HỆ THỐNG DỰ BÁO TẮC NGHẼN GIAO THÔNG ĐÔ THỊ")
print("  Apache Spark Distributed Pipeline (Đã tích hợp Deep Outlier Removal)")
print("  Mô hình: GBTRegressor | Cụm: 1 Master + 2 Workers")
print("  Heatmap: Point-based | Segment-based | Hexagonal Binning")
print("================================================================")
print("")

# ================================================================
# 1: TẠO CẤU TRÚC THƯ MỤC LƯU KẾT QUẢ
# ================================================================
print("[CHUẨN BỊ] Tạo cấu trúc thư mục lưu trữ kết quả tự động...")

# Define the base Google Drive path
base_drive_path = '/content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi'

# Define output directories relative to the base Google Drive path
output_base = os.path.join(base_drive_path, 'KetQua_HuanLuyen')
output_csv = os.path.join(output_base, 'CSV')
output_chart = os.path.join(output_base, 'BieuDo')
output_json = os.path.join(output_base, 'JSON')
models_dir = os.path.join(base_drive_path, 'Models')

for folder in [output_csv, output_chart, output_json, models_dir]:
    os.makedirs(folder, exist_ok=True)
    print(f"  + Đã tạo/kiểm tra thư mục: {folder}/")

print("[CHUẨN BỊ] Hoàn tất cấu trúc thư mục.")
print("  Ghi chú: Tất cả CSV -> {0}/, BieuDo -> {1}/, JSON -> {2}/, Models -> {3}/".format(
    output_csv, output_chart, output_json, models_dir))
print("")

# Dán Authtoken của bạn vào đây
NGROK_AUTH_TOKEN = "2Q7mq3415oIWaQOuzASAUvniTNe_4CtuKrNqfKRdzEcbmzXm1"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# ================================================================
# 2. KHỞI TẠO KẾT NỐI SPARK CLUSTER (1 MASTER + 2 WORKERS)
# ================================================================
print("================================================================")
print("  KHỞI TẠO KẾT NỐI SPARK CLUSTER TRÊN GOOGLE COLAB")
print("================================================================")
print("")
print("[BƯỚC 0.1] Đang khởi tạo SparkSession...")

# Cấu hình Cluster: 2 Workers, mỗi Worker 1 Core và 4GB RAM
# Áp dụng các giới hạn phân bổ bộ nhớ chuyên sâu
spark = SparkSession.builder \
    .appName("Traffic_Pipeline_Colab") \
    .master("local-cluster[2, 1, 4096]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.memoryOverhead", "512m") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.25") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.ui.port", "4040") \
    .config("spark.ui.showConsoleProgress", "true") \
    .getOrCreate()

# ================================================================
# 3. TẠO LINK TRUY CẬP SPARK UI QUA NGROK
# ================================================================
# Lấy thông tin Server (Master) nội bộ.
master_url = spark.sparkContext.master
print(f"Server (Master) URL: {master_url}")
try:
    # Ngắt các kết nối ngrok cũ nếu có để tránh lỗi
    ngrok.kill()
    # Mở kết nối (tunnel) mới đến cổng 4040
    public_url = ngrok.connect(4040)
    print(f"Link Spark UI Client (Worker): {public_url}")
except Exception as e:
    print(f"\n[LỖỖI] Không thể tạo link ngrok: {e}")
spark.sparkContext.setLogLevel("WARN")
print("[BƯỚC 0.2] Đã khởi tạo Spark thành công!")

# ================================================================
#  QUY TRÌNH 1/4: ĐỌC VÀ TIỀN XỬ LÝ DỮ LIỆU THÔ
# ================================================================
data_path = '/content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/Data/train.csv'
if not os.path.exists(data_path):
    print(f"[LỖI] Không tìm thấy file dữ liệu: {data_path}")
    print("  Vui lòng đảm bảo file train.csv nằm trong thư mục Data/")
    # Không dùng exit() để tránh đóng kernel Colab
else:
    from pyspark.sql.functions import to_timestamp
    print("================================================================")
    print("  QUY TRÌNH 1/4: ĐỌC VÀ TIỀN XỬ LÝ DỮ LIỆU THÔ")
    print("================================================================")
    print("")

    print("[BƯỚC 1.1] Worker 1 & 2 đọc dữ liệu song song từ Data/train.csv...")
    start_time = time.time()
    df = spark.read.csv(data_path, header=True, inferSchema=True)
    elapsed = time.time() - start_time
    raw_count = df.count()
    print(f"  [OK] Đã đọc xong trong {elapsed:.1f}s. Tổng số dòng: {raw_count}")
    print("")

    # --- Chuyển đổi kiểu dữ liệu ---
    print("[BƯỚC 1.1.0] Chuyển đổi kiểu dữ liệu Timestamp...")
    df = df.withColumn("pickup_datetime", to_timestamp(col("pickup_datetime")))
    print("  [OK] Đã chuyển đổi pickup_datetime sang kiểu Timestamp để trích xuất đặc trưng.")
    print("")

    # --- Xử lý dữ liệu thiếu (Null) ---
    print("[BƯỚC 1.1.1] Xử lý dữ liệu thiếu (Null Values)...")
    initial_count_before_null_drop = df.count()
    df = df.na.drop()
    count_after_null_drop = df.count()
    null_rows_removed = initial_count_before_null_drop - count_after_null_drop
    print(f"  [OK] Đã loại bỏ {null_rows_removed} hàng chứa giá trị Null. Dữ liệu còn lại: {count_after_null_drop} dòng.")
    raw_count = count_after_null_drop
    print("")

    print("[BƯỚC 1.2] Hiển thị 10 dòng đầu tiên (Worker -> Master via Collect):")
    df.show(10)

    print("[BƯỚC 1.3] Làm sạch dữ liệu cơ bản (Data Cleaning)...")
    # CẬP NHẬT: Thêm điều kiện passenger_count [1-6], giới hạn trip_duration <= 7200, và Longitude <= -73.6
    df_clean = df.filter(
        (col("trip_duration") >= 60) & (col("trip_duration") <= 7200) \
        & (col("passenger_count") >= 1) & (col("passenger_count") <= 6) \
        & (col("pickup_longitude") >= -74.3) & (col("pickup_longitude") <= -73.6) \
        & (col("pickup_latitude") >= 40.5) & (col("pickup_latitude") <= 41.0)
    )
    clean_count = df_clean.count()
    removed = raw_count - clean_count
    print(f"  [OK] Trước: {raw_count} dòng -> Sau khi lọc tọa độ ở New York: {clean_count} dòng (Loại bỏ {removed} dòng nhiễu thô)")
    print("")

    print("[BƯỚC 1.4] Kỹ thuật đặc trưng tăng cường dữ liệu (Feature Engineering Data)...")
    df_features = df_clean.withColumn("hour", expr("hour(pickup_datetime)")) \
                          .withColumn("minute", expr("minute(pickup_datetime)")) \
                          .withColumn("time_bin_15m", expr("hour * 4 + floor(minute / 15)")) \
                          .withColumn("day_of_week", expr("dayofweek(pickup_datetime)")) \
                          .withColumn("month", expr("month(pickup_datetime)")) \
                          .withColumn("log_trip_duration", expr("log(trip_duration + 1)")) # Tuân thủ chính xác công thức log(trip_duration + 1)

    df_features = df_features.withColumn("hour_sin", sin(2 * math.pi * col("hour") / 24)) \
                            .withColumn("hour_cos", cos(2 * math.pi * col("hour") / 24))
    df_features = df_features.withColumn("lon1", radians(col("pickup_longitude"))) \
                            .withColumn("lat1", radians(col("pickup_latitude"))) \
                            .withColumn("lon2", radians(col("dropoff_longitude"))) \
                            .withColumn("lat2", radians(col("dropoff_latitude"))) \
                            .withColumn("dlon", col("lon2") - col("lon1")) \
                            .withColumn("dlat", col("lat2") - col("lat1")) \
                            .withColumn("a", sin(col("dlat")/2)**2 + cos(col("lat1")) * cos(col("lat2")) * sin(col("dlon")/2)**2) \
                            .withColumn("distance_km", 2 * 6371 * asin(sqrt(col("a"))))

    # Thêm cột vĩ độ trung bình để scale kinh độ.
    df_features = df_features.withColumn("mean_lat", radians((col("pickup_latitude") + col("dropoff_latitude")) / 2))
    # Tính Manhattan distance theo km
    df_features = df_features.withColumn(
        "manhattan_distance_km",
        (spark_abs(col("pickup_latitude") - col("dropoff_latitude")) * 111.045) +
        (spark_abs(col("pickup_longitude") - col("dropoff_longitude")) * 111.045 * cos(col("mean_lat")))
    ).drop("mean_lat")
    # Tính tốc độ thực tế của mỗi chuyến đi (km/h)
    df_features = df_features.withColumn("speed_kmh", expr("distance_km / (trip_duration / 3600)"))
    # Tính vận tốc tự do (v_max) dựa trên dữ liệu lịch sử (Lấy phân vị 95% của tốc độ để loại bỏ nhiễu)
    # Lưu ý: approxQuantile trả về list, lấy phần tử đầu tiên
    v_max_val = df_features.approxQuantile("speed_kmh", [0.95], 0.01)[0]
    print(f"  [INFO] Vận tốc tự do (v_max) ước tính từ dữ liệu (95th percentile): {v_max_val:.2f} km/h")

    # Tính tỷ lệ tắc nghẽn R = (v_max - v_thực_tế) / v_max
    # Hoặc R = v_thực_tế / v_max (tùy định nghĩa của bạn). Dưới đây dùng: R = v_thực_tế / v_max
    # Đảm bảo v_max > 0 để tránh lỗi chia cho 0
    from pyspark.sql.functions import lit
    df_features = df_features.withColumn("v_max", lit(v_max_val))
    df_features = df_features.withColumn("congestion_ratio_R", expr("speed_kmh / v_max"))
    df_features = df_features.drop("minute", "lon1", "lat1", "lon2", "lat2", "dlon", "dlat", "a")

    # Thêm StringIndexer cho biến store_and_fwd_flag theo đúng yêu cầu
    indexer_vendor = StringIndexer(inputCol="vendor_id", outputCol="vendor_id_idx")
    df_features = indexer_vendor.fit(df_features).transform(df_features)
    indexer_flag = StringIndexer(inputCol="store_and_fwd_flag", outputCol="store_and_fwd_flag_idx")
    df_features = indexer_flag.fit(df_features).transform(df_features)
    encoder_vendor = OneHotEncoder(inputCols=["vendor_id_idx"], outputCols=["vendor_id_ohe"])
    df_features = encoder_vendor.fit(df_features).transform(df_features)
    indexer_day = StringIndexer(inputCol="day_of_week", outputCol="day_of_week_idx")
    df_features = indexer_day.fit(df_features).transform(df_features)
    encoder_day = OneHotEncoder(inputCols=["day_of_week_idx"], outputCols=["day_of_week_ohe"])
    df_features = encoder_day.fit(df_features).transform(df_features)
    print("  [OK] Đã tạo thành công tất cả các đặc trưng mới.")
    print("")

    # ===================================
    # XỬ LÝ OUTLIER CHUYÊN SÂU
    # ===================================
    print("[BƯỚC 1.5] Xử lý Ngoại lệ (Outliers)...")
    # 1. Lọc theo khoảng cách tối thiểu
    df_features = df_features.filter(col("distance_km") > 0.1)
    # 2. Lọc theo tốc độ vật lý hợp lý (2 km/h đến 100 km/h)
    df_features = df_features.withColumn("temp_speed_kmh", col("distance_km") / (col("trip_duration") / 3600))
    df_features = df_features.filter((col("temp_speed_kmh") >= 2) & (col("temp_speed_kmh") <= 100))
    # 3. Loại bỏ bằng IQR cho trip_duration
    quantiles_dur = df_features.approxQuantile("trip_duration", [0.25, 0.75], 0.01)
    q1_dur, q3_dur = quantiles_dur[0], quantiles_dur[1]
    iqr_dur = q3_dur - q1_dur
    df_features = df_features.filter(
        (col("trip_duration") >= (q1_dur - 1.5 * iqr_dur)) &
        (col("trip_duration") <= (q3_dur + 1.5 * iqr_dur))
    )
    # 4. Loại bỏ bằng IQR cho distance_km
    quantiles_dist = df_features.approxQuantile("distance_km", [0.25, 0.75], 0.01)
    q1_dist, q3_dist = quantiles_dist[0], quantiles_dist[1]
    iqr_dist = q3_dist - q1_dist
    df_features = df_features.filter(
        (col("distance_km") >= (q1_dist - 1.5 * iqr_dist)) &
        (col("distance_km") <= (q3_dist + 1.5 * iqr_dist))
    )
    df_features = df_features.drop("temp_speed_kmh")
    final_clean_count = df_features.count()
    outliers_removed = clean_count - final_clean_count
    print(f"  [OK] Đã loại bỏ thêm {outliers_removed} điểm dữ liệu ngoại lệ (Outliers).")
    print(f"  [OK] Dữ liệu sạch còn lại để huấn luyện: {final_clean_count} dòng.")
    print("")
    # --- LƯU CSV NHÓM 1: DỮ LIỆU ĐẦU VÀO & TIỀN XỬ LÝ ---
    print("[LƯU TRỮ] Xuất CSV Nhóm 1: Thông tin làm sạch dữ liệu...")
    summary_data = pd.DataFrame([
        {"Stage": "Before Cleaning", "Count": initial_count_before_null_drop}, # Cập nhật count gốc
        {"Stage": "After Null Drop", "Count": raw_count},
        {"Stage": "After Basic Cleaning", "Count": clean_count},
        {"Stage": "After Outlier Removal (High R2)", "Count": final_clean_count}
    ])
    summary_data.to_csv(os.path.join(models_dir, '1_2_cleaning_summary.csv'), index=False, sep=',')
    summary_data.to_csv(f'{output_csv}/1_2_cleaning_summary.csv', index=False, sep=',')
    print(f"  -> Đã lưu: {output_csv}/1_2_cleaning_summary.csv")
    print("")

    # ================================================================
    #  QUY TRÌNH 2/4: PHÂN TÍCH EDA VÀ BIỂU ĐỒ (100% DỮ LIỆU - NO SAMPLING)
    # ================================================================
    print("================================================================")
    print("  QUY TRÌNH 2/4: PHÂN TÍCH EDA + VẼ 5 BIỂU ĐỒ CHUẨN TÀI LIỆU")
    print("================================================================")
    print("")

    # ----------------------------------------------------------------
    # CHIẾN LƯỢC: TÍNH TOÁN CÁC BIẾN MỚI TRỰC TIẾP TRÊN SPARK WORKERS
    # ----------------------------------------------------------------
    print("[BƯỚC 2.1] Tính toán đặc trưng phụ trợ trực tiếp trên Spark Cluster...")
    # Tính tốc độ km/h và làm tròn tọa độ trên Cluster (Bảo vệ Driver 2GB RAM)
    df_features = df_features.withColumn("speed_kmh", col("distance_km") / (col("trip_duration") / 3600)) \
                             .withColumn("lat_r", spark_round(col("pickup_latitude"), 3)) \
                             .withColumn("lon_r", spark_round(col("pickup_longitude"), 3))
    total_records = df_features.count()
    print(f"  [OK] Đang phân tích trên toàn bộ dữ liệu sạch: {total_records} bản ghi.\n")

    # ----------------------------------------------------------------
    # Tổng hợp dữ liệu (Aggregation) trực tiếp trên Spark cho Bar Chart
    # ----------------------------------------------------------------
    hourly_counts = df_features.groupBy("hour").count().orderBy("hour").toPandas()
    print("[BƯỚC 2.2] Vẽ 5 biểu đồ EDA (Exploratory Data Analysis)....")

    # ----------------------------------------------------------------
    # 1. Phân phối tốc độ di chuyển (Speed Distribution Histogram)
    # ----------------------------------------------------------------
    print("  Vẽ biểu đồ 1: Phân phối tốc độ trung bình (Speed Distribution)...")
    pdf_speed = df_features.select("speed_kmh").toPandas()
    plt.figure(figsize=(10, 6))
    plt.hist(pdf_speed['speed_kmh'], bins=60, color='mediumpurple', edgecolor='black')
    plt.axvline(x=pdf_speed['speed_kmh'].mean(), color='red', linestyle='--', linewidth=2, label='Tốc độ trung bình')
    plt.title('1. Phân phối tốc độ trung bình toàn chuyến đi')
    plt.xlabel('Vận tốc (km/h)')
    plt.ylabel('Tần suất (Số lượng chuyến đi)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{output_chart}/eda_1_speed_histogram.png', dpi=150)
    plt.close()
    del pdf_speed; gc.collect() # Giải phóng RAM

    # ----------------------------------------------------------------
    # 2. Tốc độ trung bình theo giờ (Line Plot with Std Dev)
    # ----------------------------------------------------------------
    print("  Vẽ biểu đồ 2: Tốc độ trung bình theo 24 giờ (Line Plot with Std Dev)...")
    # Tính Mean và StdDev trực tiếp trên Spark để tối ưu RAM
    hourly_speed_df = df_features.groupBy("hour").agg(
        spark_mean("speed_kmh").alias("mean_speed"),
        spark_stddev("speed_kmh").alias("std_speed")
    ).orderBy("hour").toPandas()

    plt.figure(figsize=(12, 6))
    plt.plot(hourly_speed_df['hour'], hourly_speed_df['mean_speed'], color='skyblue', marker='o', linewidth=2, label='Tốc độ trung bình')
    # Vẽ dải biến thiên (Độ lệch chuẩn)
    plt.fill_between(hourly_speed_df['hour'],
                     hourly_speed_df['mean_speed'] - hourly_speed_df['std_speed'],
                     hourly_speed_df['mean_speed'] + hourly_speed_df['std_speed'],
                     color='skyblue', alpha=0.3, label='Độ lệch chuẩn (Mức độ biến động)')
    plt.title('2. Tốc độ trung bình theo 24 giờ và Sự biến động tắc nghẽn')
    plt.xlabel('Giờ trong ngày (0 - 23h)')
    plt.ylabel('Vận tốc (km/h)')
    plt.xticks(range(0, 24))
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{output_chart}/eda_2_hourly_speed_line.png', dpi=150)
    plt.close()

    # ----------------------------------------------------------------
    # 3. Speed vs Trip Distance Analysis (3D & 2D Joint Insight)
    # ----------------------------------------------------------------
    print("  Vẽ biểu đồ 3: Phân tích Tốc độ theo Khoảng cách...")
    # 1. Trích xuất dữ liệu sang Pandas
    pdf_scatter = df_features.select("distance_km", "speed_kmh", "hour").toPandas()
    x = pdf_scatter['distance_km']
    y = pdf_scatter['speed_kmh']
    z = pdf_scatter['hour']

    # 2. Khởi tạo Figure với kích thước lớn để đảm bảo độ rõ nét
    fig = plt.figure(figsize=(22, 10))
    fig.suptitle('PHÂN TÍCH MỐI TƯƠNG QUAN TỐC ĐỘ VÀ KHOẢNG CÁCH', fontsize=20, fontweight='bold', y=0.98)
    # --- BIỂU ĐỒ 3A: 3D Scatter Plot (Góc nhìn đa chiều) ---
    # Thêm trục thời gian (z) để xem khung giờ ảnh hưởng thế nào đến tốc độ khi đi xa
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    scatter = ax1.scatter(x, z, y, c=z, cmap='turbo', alpha=0.2, edgecolors='none', s=4)
    ax1.set_title('3A. Phân tán Khoảng cách - Giờ - Tốc độ', pad=20, fontsize=15)
    ax1.set_xlabel('Khoảng cách (km)', labelpad=10)
    ax1.set_ylabel('Giờ trong ngày', labelpad=10)
    ax1.set_zlabel('Tốc độ (km/h)', labelpad=10)
    # Thêm thanh màu để nhận diện khung giờ
    cbar = fig.colorbar(scatter, ax=ax1, pad=0.1, shrink=0.6)
    cbar.set_label('Thanh màu: Khung giờ (0h - 23h)')

    # --- BIỂU ĐỒ 3B: 2D Scatter + Regression Line (Phân tích xu hướng) ---
    ax2 = fig.add_subplot(1, 2, 2)
    # Vẽ Scatter với alpha thấp để mô phỏng mật độ của Joint Plot
    ax2.scatter(x, y, alpha=0.1, color='teal', s=8, label='Dữ liệu chuyến đi')
    # Tính toán đường hồi quy tuyến tính: $y = mx + b$
    m, b = np.polyfit(x, y, 1)
    ax2.plot(x, m*x + b, color='red', linewidth=3, label=f'Đường hồi quy (Hệ số góc: {m:.2f})')
    # Cấu hình trục và tiêu đề theo yêu cầu văn bản
    ax2.set_title('3B. Xu hướng Tốc độ theo Quãng đường (2D)', pad=20, fontsize=15)
    ax2.set_xlabel('Khoảng cách (km)', fontsize=12)
    ax2.set_ylabel('Tốc độ (km/h)', fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.5)
    ax2.legend(loc='upper left')
    # Thêm chú giải phân tích trực tiếp
    # analysis_text = "Phân tích: Nếu độ dốc dương (>0),\ngiả thuyết đi xa có tốc độ cao hơn là chính xác."
    # ax2.text(0.05, 0.85, analysis_text, transform=ax2.transAxes, fontsize=11,
           # verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

    # 3. Hoàn thiện và lưu file
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(f'{output_chart}/eda_3_speed_vs_distance_combined.png', dpi=150, bbox_inches='tight')
    plt.close()
    # Giải phóng bộ nhớ
    del pdf_scatter; gc.collect()

    # ----------------------------------------------------------------
    # 4. Box plot: Tốc độ theo ngày trong tuần
    # ----------------------------------------------------------------
    print("  Vẽ biểu đồ 4: Boxplot Tốc độ theo ngày trong tuần...")
    pdf_box = df_features.select("day_of_week", "speed_kmh").toPandas()
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='day_of_week', y='speed_kmh', data=pdf_box, hue='day_of_week', palette='Set2', legend=False)
    plt.title('4. Boxplot: Phân phối tốc độ theo ngày trong tuần (So sánh ngày làm việc vs ngày nghỉ)')
    plt.xlabel('Thứ trong tuần')
    plt.ylabel('Tốc độ (km/h)')
    plt.tight_layout()
    plt.savefig(f'{output_chart}/eda_4_boxplot_speed_day.png', dpi=150)
    plt.close()
    del pdf_box; gc.collect()

    # ----------------------------------------------------------------
    # 5. Ma trận tương quan (Correlation Heatmap)
    # ----------------------------------------------------------------
    print("  Vẽ biểu đồ 5: Ma trận tương quan (Correlation Heatmap)...")
    # Tự động chọn các biến có sẵn trong dataframe. (Thêm biến thời tiết vào mảng này nếu Data có)
    target_cols = ["speed_kmh", "distance_km", "hour", "day_of_week",
                   "pickup_longitude", "pickup_latitude", "passenger_count"]

    # Chỉ lấy các cột thực sự tồn tại trong Spark DataFrame để tránh lỗi
    available_cols = [c for c in target_cols if c in df_features.columns]
    pdf_corr = df_features.select(available_cols).toPandas()

    plt.figure(figsize=(10, 8))
    # Tính ma trận tương quan Pearson
    corr_matrix = pdf_corr.corr(method='pearson')

    # Vẽ biểu đồ nhiệt
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1,
                square=True, linewidths=.5, cbar_kws={"shrink": 0.8})
    plt.title('5. Ma trận tương quan (Pearson Correlation Heatmap)')
    plt.tight_layout()
    plt.savefig(f'{output_chart}/eda_5_correlation_heatmap.png', dpi=150)
    plt.close()
    del pdf_corr; gc.collect()

    print(f"  [OK] Đã lưu 5 biểu đồ EDA vào thư mục {output_chart}/")

    # ----------------------------------------------------------------
    # XUẤT CSV - TÍNH TOÁN BẰNG SPARK, CHỈ PULL KẾT QUẢ CUỐI (Tối ưu I/O)
    # ----------------------------------------------------------------
    print("[LƯU TRỮ] Xuất CSV Nhóm 2: Kết quả phân tích EDA...")

    # Gom nhóm dữ liệu theo Giờ và Tọa độ
    df_hourly_real = df_features.groupBy("hour", "lat_r", "lon_r").count().orderBy("hour").toPandas()
    # Lưu thành file CSV mới dành riêng cho Heatmap 24h
    path_hourly_csv = f'{output_csv}/2_4_hourly_hotspot.csv'
    df_hourly_real.to_csv(path_hourly_csv, index=False)
    print(f"Đã lưu dữ liệu 24h tại: {path_hourly_csv}")

    time_dist = df_features.groupBy('hour').count().toPandas()
    time_dist.to_csv(f'{output_csv}/2_1_time_distribution.csv', index=False)

    speed_by_month = df_features.groupBy('month').agg(spark_mean('speed_kmh').alias('speed_kmh')).toPandas()
    speed_by_month.to_csv(f'{output_csv}/2_2_avg_speed_by_month.csv', index=False)

    speed_by_day = df_features.groupBy('day_of_week').agg(spark_mean('speed_kmh').alias('speed_kmh')).toPandas()
    speed_by_day.to_csv(f'{output_csv}/2_2_avg_speed_by_day.csv', index=False)

    hotspot_df = df_features.groupBy("pickup_latitude", "pickup_longitude").count().orderBy(desc("count")).limit(100).toPandas()
    hotspot_df.to_csv(f'{output_csv}/2_3_hotspot_density.csv', index=False)

    print(f"  [OK] Đã hoàn thành lưu biểu đồ và xuất CSV phân tích.")

    print("----------------------------------------------------------------")
    print("  TẠO 3 LOẠI HEATMAP CHUYÊN DỤNG CHO PHÂN TÍCH GIAO THÔNG")
    print("----------------------------------------------------------------\n")
    # Tọa độ trung tâm Manhattan, New York
    center_lat, center_lon = 40.7589, -73.9851
    # Vận tốc càng thấp -> congestion_ratio_R càng nhỏ -> delay_weight càng lớn
    df_features = df_features.withColumn(
        "delay_weight",
        expr("CASE WHEN 1.0 - congestion_ratio_R < 0 THEN 0.0 ELSE 1.0 - congestion_ratio_R END")
    )

    # --- [HEATMAP 1/3] Mật độ lịch sử (Historical) ---
    print("[HEATMAP 1/3] Đang tạo Heatmap mật độ lịch sử...")
    m_hist = folium.Map(location=[center_lat, center_lon], zoom_start=12,
                        tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
                        attr='Google')

    hist_data = hotspot_df[['pickup_latitude', 'pickup_longitude', 'count']].values.tolist()
    HeatMap(hist_data, radius=15, blur=10,
            gradient={0.2: 'blue', 0.4: 'lime', 0.6: 'orange', 1: 'red'}).add_to(m_hist)
    m_hist.save(f'{output_chart}/heatmap_historical.html')
    print(f"  [OK] Đã lưu: {output_chart}/heatmap_historical.html\n")


    # --- [HEATMAP 2/3] Diễn biến thời gian (HeatmapWithTime) ---
    print("[HEATMAP 2/3] Đang tạo Heatmap diễn biến theo 24 giờ...")
    m_time = folium.Map(location=[center_lat, center_lon], zoom_start=12,
                        tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
                        attr='Google')
    time_index = [f"{h:02d}:00" for h in range(24)]
    heat_data_time = []
    max_count_time = df_hourly_real['count'].max()
    for h in range(24):
        df_hour = df_hourly_real[df_hourly_real['hour'] == h]
        step_data = []
        for _, row in df_hour.iterrows():
            step_data.append([row['lat_r'], row['lon_r'], row['count'] / max_count_time])
        heat_data_time.append(step_data)
    HeatMapWithTime(heat_data_time, index=time_index, radius=10, auto_play=True).add_to(m_time)
    m_time.save(f'{output_chart}/heatmap_time_series.html')
    print(f"  [OK] Đã lưu: {output_chart}/heatmap_time_series.html\n")


    # --- [HEATMAP 3/3] DualMap (Đối soát Tốc độ vs Kẹt xe) ---
    print("[HEATMAP 3/3] Đang tạo DualMap đối soát...")
    m_dual = DualMap(location=[center_lat, center_lon], zoom_start=12, tiles=None)
    folium.TileLayer(tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}', attr='Google', name='Google Maps').add_to(m_dual.m1)
    folium.TileLayer(tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}', attr='Google', name='Google Maps').add_to(m_dual.m2)
    data_real = hotspot_df[['pickup_latitude', 'pickup_longitude', 'count']].values.tolist()
    HeatMap(data_real, radius=12, gradient={0.2: 'blue', 0.4: 'lime', 1: 'red'}).add_to(m_dual.m1)
    HeatMap(data_real, radius=12, gradient={0.4: 'yellow', 0.7: 'orange', 1: 'darkred'}).add_to(m_dual.m2)
    m_dual.save(f'{output_chart}/dualmap_traffic.html')
    print(f"  [OK] Đã lưu: {output_chart}/dualmap_traffic.html\n")

    print("=================================================================KẾT THÚC TIỀN XỬ LÝ DỮ LIỆU==================================================================")

    # ================================================================
    #  QUY TRÌNH 3/4: HUẤN LUYỆN MÔ HÌNH GBT REGRESSOR
    # ================================================================
    print("================================================================")
    print("  QUY TRÌNH 3/4: HUẤN LUYỆN MÔ HÌNH GBT REGRESSOR")
    print("================================================================")
    print("")
    feature_cols = [
        # Nhóm 1: Không gian - Quãng đường (Quan trọng nhất)
        "distance_km",
        "manhattan_distance_km",
        # Nhóm 2: Thời gian - Chu kỳ (Quyết định tình trạng kẹt xe)
        "hour_sin",
        "hour_cos",
        "day_of_week",
        "month",
        # Nhóm 3: Tọa độ điểm đầu / cuối (Nhận diện khu vực trung tâm hay ngoại ô)
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        # Nhóm 4: Đặc trưng phụ (Ít quan trọng hơn)
        "passenger_count"
    ]

    # Xử lý Categorical Features bằng StringIndexer
    indexer = StringIndexer(inputCol="vendor_id", outputCol="vendor_id_indexed")
    df_indexed = indexer.fit(df_features).transform(df_features)
    feature_cols.append("vendor_id_indexed")

    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    # Vận tốc thấp = Tắc nghẽn. Các biến thời gian/không gian sẽ chiếm ưu thế.
    ml_data = assembler.transform(df_indexed).withColumn("label", col("speed_kmh"))

    # [QUAN TRỌNG - TỐI ƯU STORAGE MEMORY ~0.8GB]
    ml_data.persist(StorageLevel.MEMORY_AND_DISK)

    # Tách dữ liệu chính xác thành 80% (Train), 20% (Test) và 20% (Val).
    train_data, val_data, test_data = ml_data.randomSplit([0.8, 0.2, 0.2], seed=42)
    print(f"  [OK] Train: {train_data.count()} | Val: {val_data.count()} | Test: {test_data.count()}\n")

    print("[BƯỚC 3.2] Huấn luyện GBTRegressor với cấu hình tối ưu...")

    # Bắt buộc cấu hình Checkpoint Directory để giải phóng Execution Memory
    spark.sparkContext.setCheckpointDir("/tmp/spark-checkpoints")
    gbt = GBTRegressor(
        featuresCol="features",
        labelCol="label",
        maxIter=100,      # Giữ 100 cây là hợp lý để mô hình học dần
        stepSize=0.1,     # Tốc độ học 0.1 là chuẩn cho GBT
        maxDepth=6,       # [QUAN TRỌNG] Giảm từ 12 xuống 6 để ngăn OOM (Vùng an toàn của RAM 4GB)
        maxBins=32,       # Khai báo rõ maxBins giúp kiểm soát dung lượng Histogram trong RAM
        checkpointInterval=10, # Cứ 10 cây thì lưu xuống ổ cứng để giải phóng bộ nhớ
        seed=42
    )
    paramGrid = (ParamGridBuilder()
                .addGrid(gbt.maxIter, [50, 100])
                .addGrid(gbt.stepSize, [0.05, 0.1])
                .build())

    # LƯU Ý: spark dùng 'rmse' để tính toán toán học, nhưng vì nhãn đã log, kết quả chính là RMSLE
    evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="label", metricName="rmse")
    cv = CrossValidator(estimator=gbt, estimatorParamMaps=paramGrid, evaluator=evaluator, numFolds=3, seed=42, parallelism=2)

    print("\n[DEBUG] BẮT ĐẦU HUẤN LUYỆN GBT - QUAN SÁT TIẾN ĐỘ TẠI ĐÂY...")
    spark.sparkContext.setLogLevel("ERROR")

    import threading
    from IPython.display import display, HTML
    import sys

    class SparkProgressMonitor(threading.Thread):
        def __init__(self, spark_context):
            super().__init__()
            self.spark_context = spark_context
            self.stop_signal = False
            self.daemon = True
            self.display_handle = None
            self.css = "<style> .rb-container { width: 100%; background-color: #222; border-radius: 8px; padding: 2px; } .rb-bar { height: 18px; border-radius: 6px; background: linear-gradient(90deg, #ff0000, #ff7f00, #ffff00, #00ff00, #0000ff, #4b0082, #9400d3, #ff0000); background-size: 200% 200%; animation: rainbow-anim 2s linear infinite; transition: width 0.3s ease; text-align: center; color: white; font-weight: bold; font-family: monospace; font-size: 12px; line-height: 18px; white-space: nowrap; overflow: hidden; } @keyframes rainbow-anim { 0% { background-position: 100% 0%; } 100% { background-position: 0% 0%; } } </style>"
            self.total_tasks = 0
            self.completed_tasks = 0

        def run(self):
            tracker = self.spark_context.statusTracker()
            sys.stdout.write("\r Đang khởi tạo mô hình và kết nối dữ liệu...")
            sys.stdout.flush()

            while not self.stop_signal:
                active_jobs = tracker.getActiveJobsIds()
                if active_jobs:
                    self.total_tasks = 0
                    self.completed_tasks = 0
                    for jid in active_jobs:
                        jinfo = tracker.getJobInfo(jid)
                        if jinfo:
                            for sid in jinfo.stageIds:
                                sinfo = tracker.getStageInfo(sid)
                                if sinfo:
                                    self.total_tasks += sinfo.numTasks
                                    self.completed_tasks += sinfo.numCompletedTasks

                    if self.total_tasks > 0:
                        percent = min(100, int((self.completed_tasks / self.total_tasks) * 100))
                        html_content = f"{self.css}<div>⚡ Tiến độ [Stage/Job]: {self.completed_tasks}/{self.total_tasks} Tasks ({percent}%)</div><div class='rb-container'><div class='rb-bar' style='width: {percent}%;'></div></div>"
                        if self.display_handle is None:
                            self.display_handle = display(HTML(html_content), display_id=True)
                        else:
                            self.display_handle.update(HTML(html_content))

                time.sleep(0.5)
            if self.display_handle and self.total_tasks > 0:
                final_html = f"{self.css}<div>⚡ Hoàn tất 100% ({self.completed_tasks}/{self.total_tasks} Tasks)</div><div class='rb-container'><div class='rb-bar' style='width: 100%;'>Done!</div></div>"
                self.display_handle.update(HTML(final_html))

    train_start_time = time.time()
    monitor = SparkProgressMonitor(spark.sparkContext)
    monitor.start()

    cvModel = cv.fit(train_data)

    monitor.stop_signal = True
    monitor.join()

    # Dọn dẹp cache
    ml_data.unpersist()

    sys.stdout.write("\r Hoàn tất toàn bộ CrossValidation (3 Folds)!\n")
    sys.stdout.flush()

    spark.sparkContext.setLogLevel("WARN")
    print("[DEBUG] ĐÃ HUẤN LUYỆN XONG VÀ TÌM ĐƯỢC MÔ HÌNH TỐI ƯU!\n")

    best_model = cvModel.bestModel
    train_elapsed = time.time() - train_start_time
    print(f"  [OK] Huấn luyện hoàn tất trong {train_elapsed:.0f}s ({train_elapsed/60:.1f} phút)\n")

    print("[BƯỚC 3.3] Đánh giá trên tập Test (Dự báo Vận tốc)...")
    predictions = best_model.transform(test_data)

    # 1. TÍNH CHỈ SỐ CHO VẬN TỐC (Biến mục tiêu chính)
    evaluator_speed = RegressionEvaluator(predictionCol="prediction", labelCol="label")
    rmse_speed = evaluator_speed.evaluate(predictions, {evaluator_speed.metricName: "rmse"})
    mae_speed = evaluator_speed.evaluate(predictions, {evaluator_speed.metricName: "mae"})
    r2_speed = evaluator_speed.evaluate(predictions, {evaluator_speed.metricName: "r2"})

    # 2. QUY ĐỔI NGƯỢC RA THỜI GIAN ĐỂ ĐÁNH GIÁ (Dùng công thức Vật lý)
    # prediction ở đây là speed_kmh. Đảm bảo speed dự báo > 0 để tránh lỗi chia 0
    from pyspark.sql.functions import when
    predictions = predictions.withColumn(
        "safe_pred_speed", when(col("prediction") <= 0.1, 0.1).otherwise(col("prediction"))
    )
    # Thời gian (giây) = (Quãng đường / Vận tốc) * 3600
    predictions = predictions.withColumn("pred_duration_sec", expr("(distance_km / safe_pred_speed) * 3600"))
    predictions = predictions.withColumn("actual_duration_sec", col("trip_duration"))

    evaluator_time = RegressionEvaluator(predictionCol="pred_duration_sec", labelCol="actual_duration_sec")
    mae_sec = evaluator_time.evaluate(predictions, {evaluator_time.metricName: "mae"})

    print("  [KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH DỰ BÁO TẮC NGHẼN]")
    print(f"  - RMSE (Vận tốc)      : {rmse_speed:.4f} km/h")
    print(f"  - MAE  (Vận tốc)      : {mae_speed:.2f} km/h")
    print(f"  - MAE  (Thời gian)    : {mae_sec:.2f} giây")
    print(f"  - R2   (Hệ số xác định): {r2_speed:.4f}\n")

    print("[LƯU TRỮ] Xuất CSV & JSON Nhóm 3-4: Kết quả huấn luyện...")
    # Cập nhật lại tên biến trong DataFrame xuất file
    results_df = pd.DataFrame([
        {"Metric": "RMSE_Speed_kmh", "Value": rmse_speed},
        {"Metric": "MAE_Speed_kmh", "Value": mae_speed},
        {"Metric": "MAE_Duration_sec", "Value": mae_sec},
        {"Metric": "R2", "Value": r2_speed}
    ])
    results_df.to_csv(os.path.join(models_dir, 'model_results.csv'), index=False, sep=',')
    results_df.to_csv(os.path.join(models_dir, '3_2_model_metrics.csv'), index=False, sep=',')
    results_df.to_csv(f'{output_csv}/model_results.csv', index=False, sep=',')
    results_df.to_csv(f'{output_csv}/3_2_model_metrics.csv', index=False, sep=',')
    results_df.to_json(os.path.join(models_dir, '3_2_model_metrics.json'), orient='records')
    results_df.to_json(f'{output_json}/3_2_model_metrics.json', orient='records')

    best_params_dict = {p.name: v for p, v in best_model.extractParamMap().items()}
    best_params_clean = {k: v for k, v in best_params_dict.items() if isinstance(v, (int, float, str, bool))}
    final_results_json = {
            "Metrics": {
                "RMSE_Speed": rmse_speed,
                "MAE_Speed": mae_speed,
                "MAE_Duration_sec": mae_sec,
                "R2": r2_speed
            },
            "BestParameters": best_params_clean
        }
    with open(os.path.join(models_dir, '4_2_final_results.json'), 'w', encoding='utf-8') as f:
        json.dump(final_results_json, f, indent=4)
    with open(f'{output_json}/4_2_final_results.json', 'w', encoding='utf-8') as f:
        json.dump(final_results_json, f, indent=4)

    importances = best_model.featureImportances.toArray()
    imp_df = pd.DataFrame({'Feature': feature_cols, 'Importance': importances})
    imp_df = imp_df.sort_values('Importance', ascending=False)
    imp_df.to_csv(os.path.join(models_dir, '3_3_feature_importance.csv'), index=False, sep=',')
    imp_df.to_csv(f'{output_csv}/3_3_feature_importance.csv', index=False, sep=',')

    print(f"  -> Đã lưu CSV:  {output_csv}/model_results.csv, 3_2_*.csv, 3_3_*.csv")
    print(f"  -> Đã lưu JSON: {output_json}/3_2_*.json, 4_2_*.json")

    best_model.write().overwrite().save(os.path.join(models_dir, "gbt_model"))
    print(f"  -> Đã lưu Model: {models_dir}/rf_model/\n")

    # ================================================================
    #  QUY TRÌNH 4/4: ĐÁNH GIÁ MÔ HÌNH VÀ VẼ 6 BIỂU ĐỒ (DỰ BÁO TỐC ĐỘ)
    # ================================================================
    print("================================================================")
    print("  QUY TRÌNH 4/4: ĐÁNH GIÁ MÔ HÌNH VÀ VẼ 6 BIỂU ĐỒ ĐÁNH GIÁ")
    print("================================================================")
    print("")

    print("[BƯỚC 4.1] Tính phần dư (Residual) và chuẩn bị dữ liệu...")
    predictions = predictions.withColumn("residual", col("label") - col("prediction"))

    # Lấy mẫu 10% dữ liệu để vẽ biểu đồ cho nhẹ RAM (LƯU Ý: Phải select thêm 'hour' cho PDP plot)
    preds_sample = predictions.sample(fraction=0.1, seed=42)
    # Giả sử cột giờ tên là 'hour', nếu tên khác bạn tự đổi nhé
    preds_pd = preds_sample.select("label", "prediction", "residual", "hour").toPandas()

    avp_df = preds_pd[['label', 'prediction']].rename(columns={'label': 'Actual', 'prediction': 'Predicted'})
    avp_df.to_csv(os.path.join(output_csv, '4_1_actual_vs_predicted.csv'), index=False)
    print(f"  -> Đã lưu: {output_csv}/4_1_actual_vs_predicted.csv\n")

    print("[BƯỚC 4.2] Vẽ 6 biểu đồ đánh giá mô hình GBTRegressor...")

    # =====================================================================
    # BIỂU ĐỒ 1: LEARNING CURVE (THEO SAMPLE SIZE)
    # =====================================================================
    print("  Vẽ biểu đồ 1: Learning Curve (Kiểm tra Overfitting/Underfitting)...")
    fractions = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    train_errors, val_errors = [], []

    try:
        val_data.persist()
        for frac in fractions:
            sampled_train = train_data.sample(withReplacement=False, fraction=frac, seed=42)
            sampled_train.persist()

            temp_model = gbt.fit(sampled_train)

            train_preds = temp_model.transform(sampled_train)
            train_errors.append(evaluator_speed.evaluate(train_preds))

            val_preds = temp_model.transform(val_data)
            val_errors.append(evaluator_speed.evaluate(val_preds))

            sampled_train.unpersist()
            del temp_model, train_preds, val_preds
            gc.collect()

        plt.figure(figsize=(10, 6))
        x_percent = [int(f * 100) for f in fractions]

        plt.plot(x_percent, train_errors, marker='s', color='green', linewidth=2.5, markersize=8, label='Train Error (RMSE)')
        plt.plot(x_percent, val_errors, marker='^', color='orange', linewidth=2.5, markersize=8, label='Validation Error (RMSE)')

        plt.title('1. Learning Curve: Đánh giá Overfitting/Underfitting', fontsize=14)
        plt.xlabel('Kích thước tập dữ liệu huấn luyện (%)', fontsize=12)
        plt.ylabel('Sai số RMSE (km/h)', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend(loc='best')
        plt.tight_layout()
        plt.savefig(f'{output_chart}/eval_1_learning_curve.png', dpi=150)
        plt.close()
        val_data.unpersist()
    except Exception as e:
        print(f"    [CẢNH BÁO] Không thể vẽ biểu đồ 1: {e}")

    # =====================================================================
    # BIỂU ĐỒ 2: FEATURE IMPORTANCE
    # =====================================================================
    print("  Vẽ biểu đồ 2: Feature Importance (Tầm quan trọng đặc trưng)...")
    plt.figure(figsize=(10, 8))
    imp_df_sorted = imp_df.sort_values('Importance', ascending=True)

    plt.barh(imp_df_sorted['Feature'], imp_df_sorted['Importance'], color='coral', edgecolor='black')
    plt.title('2. Feature Importance: Các biến thống trị dự báo tốc độ', fontsize=14)
    plt.xlabel('Mức độ đóng góp', fontsize=12)
    plt.ylabel('Đặc trưng (Features)', fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.savefig(f'{output_chart}/eval_2_feature_importance.png', dpi=150)
    plt.close()

    # =====================================================================
    # BIỂU ĐỒ 3: ACTUAL VS PREDICTED SPEED (Tập trung 0-15 km/h)
    # =====================================================================
    print("  Vẽ biểu đồ 3: Dự báo vs Thực tế (Actual vs Predicted Speed)...")
    plt.figure(figsize=(8, 8))
    sns.regplot(x='label', y='prediction', data=preds_pd,
                scatter_kws={'alpha': 0.3, 'color': 'teal', 's': 10},
                line_kws={'color': 'orange'})

    max_val = max(preds_pd['label'].max(), preds_pd['prediction'].max())
    plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Đường y = x')

    # Highlight vùng kẹt xe 0-15 km/h
    plt.axvspan(0, 15, color='red', alpha=0.1, label='Vùng tắc nghẽn (0-15 km/h)')

    plt.title('3. Actual Speed vs Predicted Speed', fontsize=14)
    plt.xlabel('Tốc độ thực tế (km/h)', fontsize=12)
    plt.ylabel('Tốc độ dự báo (km/h)', fontsize=12)
    plt.legend()

    # Thêm Quote nhấn mạnh
    # quote = "Mô hình có khả năng bắt được các tình huống\ntắc nghẽn nghiêm trọng (speed thấp) cực kỳ nhạy bén."
    # plt.text(2, max_val*0.85, quote, fontsize=10, style='italic', bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))

    plt.tight_layout()
    plt.savefig(f'{output_chart}/eval_3_actual_vs_predicted.png', dpi=150)
    plt.close()

    # =====================================================================
    # BIỂU ĐỒ 4: RESIDUAL PLOT BY SPEED RANGE
    # =====================================================================
    print("  Vẽ biểu đồ 4: Biểu đồ phần dư theo mức vận tốc (Residual Plot)...")
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x='label', y='residual', data=preds_pd, alpha=0.3, color='purple')

    plt.axhline(0, color='red', linestyle='--', linewidth=2)
    plt.title('4. Residual Plot by Speed Range (Phân tích sai số)', fontsize=14)
    plt.xlabel('Tốc độ thực tế (km/h)', fontsize=12)
    plt.ylabel('Phần dư (Thực tế - Dự báo)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.savefig(f'{output_chart}/eval_4_residual_by_speed.png', dpi=150)
    plt.close()

    # =====================================================================
    # BIỂU ĐỒ 5: PARTIAL DEPENDENCE PLOT (PDP) - TIME OF DAY
    # =====================================================================
    print("  Vẽ biểu đồ 5: Partial Dependence Plot (Tác động của Giờ lên Tốc độ)...")
    plt.figure(figsize=(10, 6))

    # Do PDP phức tạp trên Spark, ta dùng Marginal Mean Prediction theo từng giờ làm xấp xỉ
    if 'hour' in preds_pd.columns:
        pdp_data = preds_pd.groupby('hour')['prediction'].mean().reset_index()
        sns.lineplot(x='hour', y='prediction', data=pdp_data, marker='o', color='crimson', linewidth=2.5, markersize=8)

        plt.title('5. Partial Dependence Plot (PDP): Tác động của Giờ lên Tốc độ', fontsize=14)
        plt.xlabel('Giờ trong ngày (0 - 23h)', fontsize=12)
        plt.ylabel('Tốc độ dự báo trung bình (km/h)', fontsize=12)
        plt.xticks(range(0, 24))
        plt.grid(True, linestyle='--', alpha=0.6)

        # Đánh dấu thung lũng 8h và 17h
        plt.axvline(8, color='gray', linestyle=':', label='Giờ cao điểm sáng (8h)')
        plt.axvline(17, color='gray', linestyle='--', label='Giờ cao điểm chiều (17h)')
        plt.legend()
    else:
        print("    [LỖI] Không tìm thấy cột 'hour' để vẽ PDP.")

    plt.tight_layout()
    plt.savefig(f'{output_chart}/eval_5_pdp_time_of_day.png', dpi=150)
    plt.close()

    # =====================================================================
    # BIỂU ĐỒ 6: QUANTILE-QUANTILE (Q-Q) PLOT OF RESIDUALS
    # =====================================================================
    print("  Vẽ biểu đồ 6: Q-Q Plot kiểm tra phân phối chuẩn của sai số...")
    plt.figure(figsize=(8, 8))

    stats.probplot(preds_pd['residual'].dropna(), dist="norm", plot=plt)

    plt.title('6. Quantile-Quantile (Q-Q) Plot of Residuals', fontsize=14)
    plt.xlabel('Phân vị lý thuyết (Theoretical Quantiles)', fontsize=12)
    plt.ylabel('Phân vị mẫu (Sample Quantiles)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(f'{output_chart}/eval_6_qq_plot.png', dpi=150)
    plt.close()

    # Dọn dẹp RAM Driver sau khi vẽ xong
    del preds_pd
    gc.collect()
    print(f"\n[OK] Đã hoàn tất và lưu 6 biểu đồ Đánh giá vào: {output_chart}/")

    # ================================================================
    # TỔNG KẾT (Đã cập nhật theo logic dự báo Vận tốc/Tắc nghẽn)
    # ================================================================
    print("================================================================")
    print("  HOÀN TẤT TOÀN BỘ QUY TRÌNH DỰ BÁO TẮC NGHẼN!")
    print("================================================================")
    print("")
    print("  [TỔNG KẾT KẾT QUẢ SAU HUẤN LUYỆN]")
    # Thay RMSLE bằng RMSE của Vận tốc, bổ sung MAE Vận tốc
    print(f"  - RMSE Vận tốc : {rmse_speed:.4f} km/h")
    print(f"  - MAE Vận tốc  : {mae_speed:.2f} km/h")
    print(f"  - MAE Thời gian : {mae_sec:.2f}s (Quy đổi từ vận tốc)")
    print(f"  - R2 Score     : {r2_speed:.4f}")
    print("")
    print(f"  - Thời gian huấn luyện (3 Folds): {train_elapsed:.0f}s ({train_elapsed/60:.1f} phút)")
    print("")
    print("  [THÀNH CÔNG]")


  HỆ THỐNG DỰ BÁO TẮC NGHẼN GIAO THÔNG ĐÔ THỊ
  Apache Spark Distributed Pipeline (Đã tích hợp Deep Outlier Removal)
  Mô hình: GBTRegressor | Cụm: 1 Master + 2 Workers
  Heatmap: Point-based | Segment-based | Hexagonal Binning

[CHUẨN BỊ] Tạo cấu trúc thư mục lưu trữ kết quả tự động...
  + Đã tạo/kiểm tra thư mục: /content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/KetQua_HuanLuyen/CSV/
  + Đã tạo/kiểm tra thư mục: /content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/KetQua_HuanLuyen/BieuDo/
  + Đã tạo/kiểm tra thư mục: /content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/KetQua_HuanLuyen/JSON/
  + Đã tạo/kiểm tra thư mục: /content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/Models/
[CHUẨN BỊ] Hoàn tất cấu trúc thư mục.
  Ghi chú: Tất cả CSV -> /content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/KetQua_HuanLuyen/CSV/, BieuDo -> /content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/KetQua_HuanLuyen/BieuDo/, JSON -> /content/drive/MyDrive/Du_bao_tac_duong

Link Spark UI Client (Worker): NgrokTunnel: "https://adef-34-82-76-76.ngrok-free.app" -> "http://localhost:4040"
[BƯỚC 0.2] Đã khởi tạo Spark thành công!
  QUY TRÌNH 1/4: ĐỌC VÀ TIỀN XỬ LÝ DỮ LIỆU THÔ

[BƯỚC 1.1] Worker 1 & 2 đọc dữ liệu song song từ Data/train.csv...
  [OK] Đã đọc xong trong 30.8s. Tổng số dòng: 1458644

[BƯỚC 1.1.0] Chuyển đổi kiểu dữ liệu Timestamp...
  [OK] Đã chuyển đổi pickup_datetime sang kiểu Timestamp để trích xuất đặc trưng.

[BƯỚC 1.1.1] Xử lý dữ liệu thiếu (Null Values)...
  [OK] Đã loại bỏ 0 hàng chứa giá trị Null. Dữ liệu còn lại: 1458644 dòng.

[BƯỚC 1.2] Hiển thị 10 dòng đầu tiên (Worker -> Master via Collect):
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|       id|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|store_an

 Hoàn tất toàn bộ CrossValidation (3 Folds)!
[DEBUG] ĐÃ HUẤN LUYỆN XONG VÀ TÌM ĐƯỢC MÔ HÌNH TỐI ƯU!

  [OK] Huấn luyện hoàn tất trong 2923s (48.7 phút)

[BƯỚC 3.3] Đánh giá trên tập Test (Dự báo Vận tốc)...
  [KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH DỰ BÁO TẮC NGHẼN]
  - RMSE (Vận tốc)      : 4.3500 km/h
  - MAE  (Vận tốc)      : 3.15 km/h
  - MAE  (Thời gian)    : 151.17 giây
  - R2   (Hệ số xác định): 0.5060

[LƯU TRỮ] Xuất CSV & JSON Nhóm 3-4: Kết quả huấn luyện...
  -> Đã lưu CSV:  /content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/KetQua_HuanLuyen/CSV/model_results.csv, 3_2_*.csv, 3_3_*.csv
  -> Đã lưu JSON: /content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/KetQua_HuanLuyen/JSON/3_2_*.json, 4_2_*.json
  -> Đã lưu Model: /content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/Models/rf_model/

  QUY TRÌNH 4/4: ĐÁNH GIÁ MÔ HÌNH VÀ VẼ 6 BIỂU ĐỒ ĐÁNH GIÁ

[BƯỚC 4.1] Tính phần dư (Residual) và chuẩn bị dữ liệu...
  -> Đã lưu: /content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/